In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt



In [3]:
print(bytes([0x41]))

b'A'


## Tokenization

Given a corpus of text, we want to train a tokenizer, T, to be able to split the text into an ordered list of tokens losslessly. There are two main methods which will be implemented in this notebook:

Method 1: Character-Level Tokenization
- Each character is considered it's own token. This is the simplest method to tokenize text, and results in very long sequences, and subpar performance.

Method 2: SentencePiece
- A variant of the BPE algorithm. This will be implemented in Rust.


In [4]:


class CharacterTokenizer:

    def __init__(self):
        self.char_to_idx = {}
        self.idx_to_char = {}

        for x in range(256):
            self.char_to_idx[bytes([x])] = x
            self.idx_to_char[x] = bytes([x])

    def tokenize(self, text: bytes):
        if type(text) == str:
            text = text.encode('utf-8')
            
        tokens = torch.tensor([self.char_to_idx[char] for char in text], dtype=torch.long)
        return tokens
    
    def decode(self, tokens):
        text = ''
        for token in tokens:
            text += self.idx_to_char[token]
        return text



## Multi-Head Attention

In transformer neural networks, the multihead attention blocks allow each token to attend to its context. Each multihead attention block takes in a tensor of size (batch, seq_len, d_model), X, representing the embeddings of the sequence of tokens from the previous transformer block (or the original embeddings if it's the first block), and projects them 3 times with learned weight matrices, $W_i^Q,W_i^K,W_i^V$, ($W_i^Q,W_i^K$ are (d_model, d_k) and $W_i^V$ is (d_model, d_v)) to Q, K, and V, respectively for each attention block (h attention blocks in total). An attention block takes in 3 matrices, Q (seq_len, d_k), K (seq_len, d_k), and V (seq_len, d_v) and outputs a matrix of size (seq_len, d_v). After each attention block is calculated, the results from each head are concatenated across the d_v axis (each head is (seq_len, d_v) so h heads concatenated will be (seq_len, d_v * h = d_model)).

</br>

$$
\text{Attention}(Q,K,V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V
$$

$$
\text{MultiHeadAttention} = \text{concat}(\text{head}_1,\text{head}_2,\text{head}_3,...,\text{head}_h)
$$

$$
\text{head}_i = \text{Attention}(XW_i^Q,XW_i^K,XW_i^V)
$$

## Transformer Decoders
In a decoder block, we mask the future tokens before computing the softmax, like this:


$$
\text{DecoderAttention}(Q,K,V) = \text{softmax}(\frac{QK^T + \text{Mask}}{\sqrt{d_k}})V
$$

where $\text{Mask}_{ij} = \begin{cases}
-\infty & \text{if } j > i \\
0 & \text{otherwise}
\end{cases}$

</br>

Example (if seq_len = 4):
</br>
$\text{Mask} = \begin{bmatrix}
    0 & -\infty & -\infty & -\infty \\
    0 & 0 & -\infty & -\infty \\
    0 & 0 & 0 & -\infty \\
    0 & 0 & 0 & 0 \\
\end{bmatrix}$


In [ ]:

class Attention(nn.Module):
    def __init__(self,d_k,d_v):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v
        self.scale = np.sqrt(d_k)

    def forward(self,Q,K,V):
        x = torch.matmul(Q, torch.transpose(K,-2,-1)) / self.scale

        # softmax over each row (a row, i, represents the "relevance" that each token, j, in the sequence has with token i)
        # creates the attention matrix
        x = F.softmax(x, -2)

        # for each token in position i, the new token in position i, after applying attention is the weighted sum of all tokens, j, using the weights from row i of the attention matrix
        x = torch.matmul(x, V)
        return x

class DecoderAttention(nn.Module):
    def __init__(self,d_k,d_v):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v
        self.scale = np.sqrt(d_k)

    def forward(self,Q,K,V):
        seq_len = Q.shape[-2]

        # torch.full creates a matrix of size n x n filled with a given element. torch.triu takes in a matrix and returns a matrix of the same size with all but elements above the main diagonal set to zero.
        # creates the attention mask that prevents future tokens from influencing past tokens
        mask = torch.triu(torch.full((seq_len, seq_len), -np.inf))

        x = (torch.matmul(Q, torch.transpose(K,-2,-1)) + mask) / self.scale

        # softmax over each row (a row, i, represents the "relevance" that each token, j, in the sequence has with token i)
        # creates the attention matrix
        x = F.softmax(x, -2)

        # for each token in position i, the new token in position i, after applying attention is the weighted sum of all tokens, j, using the weights from row i of the attention matrix
        x = torch.matmul(x, V)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,d_k,d_v,h):
        super().__init__()

        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.h = h

        self.attention_head = Attention(d_k,d_v)
        self.query_weights = nn.ModuleList([nn.Linear(in_features=d_model, out_features=d_k, bias=False) for i in range(h)])
        self.key_weights = nn.ModuleList([nn.Linear(in_features=d_model, out_features=d_k, bias=False) for i in range(h)])
        self.value_weights = nn.ModuleList([nn.Linear(in_features=d_model, out_features=d_v, bias=False) for i in range(h)])
    
    def forward(self,x):
        
        











In [ ]:
block = Attention(8,8)
block.forward(torch.randn((9,8)),torch.randn((9,8)),torch.randn((9,8)))



tensor([[0.0220, 0.0302, 0.0181, 0.0378, 0.1140, 0.0763, 0.3652, 0.1699, 0.2356],
        [0.1558, 0.0762, 0.0682, 0.0690, 0.0835, 0.0868, 0.0377, 0.0584, 0.0308],
        [0.0277, 0.0181, 0.0404, 0.0754, 0.1817, 0.0836, 0.2876, 0.1119, 0.1181],
        [0.0364, 0.1036, 0.1138, 0.0244, 0.1043, 0.0608, 0.0503, 0.1239, 0.1885],
        [0.0804, 0.0400, 0.2008, 0.0568, 0.1917, 0.0408, 0.0339, 0.1428, 0.0507],
        [0.1275, 0.0765, 0.0390, 0.2647, 0.0694, 0.0601, 0.1035, 0.0708, 0.0054],
        [0.0509, 0.0667, 0.0704, 0.0238, 0.1830, 0.0234, 0.0545, 0.2417, 0.0777],
        [0.4116, 0.2690, 0.2234, 0.3057, 0.0617, 0.0963, 0.0380, 0.0493, 0.0127],
        [0.0877, 0.3197, 0.2259, 0.1426, 0.0106, 0.4718, 0.0293, 0.0313, 0.2804]])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


tensor([[-0.3186,  0.8351,  0.8313,  0.2537,  0.2641,  0.2974,  0.6984, -0.3381],
        [-0.2071,  0.3326,  0.6053, -0.0057,  0.2798,  0.1452,  0.0516, -0.0421],
        [-0.1315,  0.6524,  0.8135,  0.1251,  0.0657,  0.3101,  0.6480, -0.3053],
        [-0.0329,  0.4218,  0.6912,  0.3749,  0.4538,  0.2798,  0.0371, -0.2523],
        [-0.0837,  0.3379,  0.6653,  0.5115,  0.2294,  0.3563,  0.0897, -0.2975],
        [-0.0325,  0.2761,  0.7820, -0.3846, -0.0187, -0.1329,  0.2769, -0.0405],
        [ 0.0200,  0.7025,  0.4143,  0.4018,  0.2439,  0.2256,  0.0845, -0.1707],
        [-0.6340,  0.2912,  1.5166, -0.2521,  0.6103, -0.0583, -0.1405, -0.1811],
        [ 0.0902, -0.1843,  1.7197,  0.1548,  0.6614,  0.8436, -0.1880,  0.6004]])